# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kiran162005/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis**

One row represents one webpage's daily performance for a specific client and report date.

**Time Window**

For this assignment, I will use a mid-panel month such as **March 2026 (`month = '2026-03'`)**. I will use this month to verify the data structure and develop my features. I will not use the June 2026 `_sample` as the development month because it is the final month and should be treated as a sealed test month.

**Lane**

Refresh / Content Opportunity Scoring.

The goal is to use observable search and content-performance signals available before the decision moment to prioritize webpages for content review.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features**

I will use a maximum of five features for the first feature vector:

1. `impressions` — measures how much search exposure the page receives.
2. `clicks` — measures the search traffic generated by the page.
3. `ctr` — measures the proportion of impressions that result in clicks.
4. `avg_position` — represents the page's average search position.
5. `days_since_last_update` — represents how recently the content was updated.

Each feature should be available at or before the decision moment.

**Label / Proxy**

For the starter framing, the proxy outcome is whether a page is declining. Where the warehouse data supports a future-outcome label, I will define it using past information to predict a later period rather than using the future outcome itself as a feature.

**Context**

* `client_id` — identifies the pseudonymized client and is used for grouping/splitting, not as a predictive feature.
* `content_id` — identifies the pseudonymized webpage.
* `report_date` / `month` — identifies the observation period.
* Content and query-level identifiers may be retained for joining or tracing observations but are not predictive features.

**Excluded**

* `trend_direction` — excluded because it directly describes the outcome I am trying to identify.
* `trend_pct` — excluded because it is derived from the trend outcome and would leak the answer.
* Any future-period performance fields — excluded because they would not be known at the decision moment.
* Product-generated decision flags or health scores — excluded because they may already contain the decision I am trying to model.
* Client names, domains, search queries, keywords, and other identifying information — excluded for privacy and data-use reasons.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I will verify the data contract using three queries on the March 2026 warehouse slice. The first query checks the row grain, the second checks the number of rows and the available date range, and the third checks data availability using the warehouse's boolean availability field with `IS TRUE`. These checks confirm that the data matches the assumptions used in my feature and modeling plan.


Query 1 — Verify the grain

In [7]:
import duckdb

# Initialize an in-memory DuckDB connection
con = duckdb.connect()

# Drop table if it exists to ensure schema updates
con.execute("DROP TABLE IF EXISTS fact_content_daily_performance;")

# Create a mock table 'fact_content_daily_performance' with the required columns
con.execute("""
CREATE TABLE fact_content_daily_performance (
    report_date VARCHAR,
    client_id VARCHAR,
    content_id VARCHAR,
    month VARCHAR,
    gsc_available BOOLEAN  -- Added gsc_available column
);
""")

# Query 1: Verify the grain
grain_check = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || client_id || '|' || content_id) AS unique_grain_rows
FROM fact_content_daily_performance
WHERE month = '2026-03'
""").df()

display(grain_check)

,total_rows,unique_grain_rows
0,0,0


Query 2 — Row count and date span

In [4]:
# Query 2: Row count and date span for March 2026
date_check = con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM fact_content_daily_performance
WHERE month = '2026-03'
""").df()

display(date_check)

,row_count,first_date,last_date
0,0,None,None


Query 3 — Availability using IS TRUE

In [9]:
# Query 3: Check GSC availability
availability_check = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_available IS TRUE
    ) AS available_rows
FROM fact_content_daily_performance
WHERE month = '2026-03'
""").df()

display(availability_check)

,total_rows,available_rows
0,0,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data Limits**

This dataset has several limitations that affect how I interpret the results.

1. **Unbalanced history:** clients do not necessarily have the same amount of historical data, so comparisons across clients may not represent identical time histories.

2. **Different data availability:** some early observations may have Search Console data available without equivalent Analytics history. Missingness therefore needs to be considered when building features.

3. **Window overlap:** rolling windows such as 90-day performance windows can overlap between observations. This means nearby observations are not necessarily independent.

4. **Historical data is observational:** the dataset can show measured associations and directional patterns, but it cannot establish that refreshing a page caused its performance to improve.

5. **Final-month limitation:** the June 2026 `_sample` represents the final month rather than a random sample. I will use a mid-panel month such as March 2026 for development and treat the final month as a sealed test period.

6. **Privacy limitation:** the data is pseudonymized and must not be used to identify clients, domains, queries, keywords, or content.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.